In [ ]:
import pandas as pd
import os, glob

all_frames = []
# IMPORTING CSV FILES FROM BRONZE LAYER
# 1. Use the ABSOLUTE path to your bronze layer folder
bronze_layer_path = r"C:\Users\medod\Desktop\Sprint one Project\Old Data (Bronze Layer)"
file_pattern = os.path.join(bronze_layer_path, "*.csv")

# 2. Check if files were actually found (to prevent the error in the future)
csv_files = glob.glob(file_pattern)
if not csv_files:
    print(f"Error: No CSV files found in {bronze_layer_path}")
    print("Please check if the path is correct.")
else:

    
    for f in csv_files:
        df = pd.read_csv(f)
        
        # Extract city and day_type from filename
        basename = os.path.basename(f).replace(".csv", "")
        parts = basename.rsplit("_", 1)  # split from right
        city = parts[0]
        day_type = "weekday" if "weekday" in parts[1] else "weekend"
        
        df["city"] = city
        df["day_type"] = day_type
        
        all_frames.append(df)

    # Combine all frames
    master = pd.concat(all_frames, ignore_index=True)
    

In [ ]:
# Drop the unnamed index column
master.drop(columns=["Unnamed: 0"], inplace=True)

# drop room_shared and room_private 
master.drop(columns=["room_shared", "room_private"], inplace=True)

In [ ]:
# Convert data typeS
master["person_capacity"] = master["person_capacity"].astype(int)

master["multi"] = master['multi'].astype(bool)
master["biz"] = master['biz'].astype(bool)

In [ ]:
# Create new columns for price per night and price per person
master["price_per_night"] = master["realSum"]/2

master["price_per_person"] = master["price_per_night"]/master["person_capacity"]
# Create a new column for host type based on the 'biz' and 'multi' columns
def host_type(row):
    if row['biz']:
        return 'Professional'
    elif row['multi']:
        return 'semi_professional'
    else:
        return 'individual'
master['host_type'] = master.apply(host_type, axis=1)
# Create a new column for location category based on the 'dist' column
master["location_category"] = pd.cut(master['dist'], bins=[0, 1, 3, 5, float('inf')], labels=['city_center', 'near_center', 'suburban', 'outskirts'])

In [ ]:
# FOR OUTLIER DETECTION (VERY HIGH PRICES)
q99 = master["realSum"].quantile(0.99)
master["is_price_outlier"] = master["realSum"] > q99

In [ ]:
#RENAME COLUMNS FOR CLARITY
master.rename(columns={
    "realSum": "total_price_2nights_eur",
    "dist": "distance_to_center_km",
    "metro_dist": "distance_to_metro_km",
    "attr_index": "attraction_index_raw",
    "attr_index_norm": "attraction_index_normalized",
    "rest_index": "restaurant_index_raw",
    "rest_index_norm": "restaurant_index_normalized",
    "lng": "longitude",
    "lat": "latitude",
}, inplace=True)

In [ ]:
# EXPORT TO CSV
master.to_csv("C:\\Users\\medod\\Desktop\\Sprint one Project\\Modified Data (silver layer)\\airbnb_europe_clean.csv", index=False)